# Spark Structured Streaming with Apache Polaris

This notebook demonstrates **real-time streaming** with Spark Structured Streaming, writing to and reading from **Iceberg tables** backed by Apache Polaris.

**What you'll learn:**
1. Streaming writes to Polaris-backed Iceberg tables
2. Incremental streaming reads from Iceberg
3. Handling partitioned tables with fanout writes
4. Maintenance: compaction, snapshot expiration

**Prerequisites:** Polaris, LocalStack, and this Spark-Jupyter container must be running.

## 1. SparkSession
The Polaris REST catalog and S3 settings are pre-configured in `spark-defaults.conf`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Polaris Streaming Demo") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Default catalog: {spark.conf.get('spark.sql.defaultCatalog')}")

## 2. Create Streaming Namespace & Tables
We create a dedicated `streaming` namespace and two Iceberg tables:
- `sensor_events` — partitioned by day, target for streaming writes
- `sensor_alerts` — target for filtered streaming output

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.streaming")
spark.sql("SHOW NAMESPACES IN polaris").show()

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.streaming.sensor_events (
        sensor_id   STRING    COMMENT 'Unique sensor identifier',
        temperature DOUBLE    COMMENT 'Temperature in Celsius',
        humidity    DOUBLE    COMMENT 'Relative humidity percentage',
        location    STRING    COMMENT 'Sensor deployment location',
        event_time  TIMESTAMP COMMENT 'Event timestamp'
    )
    USING iceberg
    PARTITIONED BY (days(event_time))
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.streaming.sensor_alerts (
        sensor_id   STRING    COMMENT 'Sensor that triggered the alert',
        temperature DOUBLE    COMMENT 'Temperature reading',
        humidity    DOUBLE    COMMENT 'Humidity reading',
        location    STRING    COMMENT 'Sensor location',
        event_time  TIMESTAMP COMMENT 'Original event timestamp',
        alert_type  STRING    COMMENT 'Type of alert triggered'
    )
    USING iceberg
""")

spark.sql("SHOW TABLES IN polaris.streaming").show()

## 3. Streaming Write — Simulated Sensor Data → Iceberg

We use Spark's `rate` source to generate a continuous stream, enrich each row with
random sensor readings, then append to our Polaris-backed Iceberg table.

The stream runs for **30 seconds** and then stops automatically.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import time

SENSORS = ["sensor-001", "sensor-002", "sensor-003", "sensor-004", "sensor-005"]
LOCATIONS = ["warehouse-A", "warehouse-B", "factory-floor", "cold-storage", "rooftop"]

# Generate a streaming DataFrame from the rate source (5 rows/sec)
raw_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Enrich with simulated sensor data
sensor_stream = (
    raw_stream
    .withColumn("sensor_id",   F.element_at(F.array(*[F.lit(s) for s in SENSORS]),   (F.col("value") % len(SENSORS) + 1).cast("int")))
    .withColumn("temperature", F.round(F.rand() * 50 + 10, 2))           # 10°C to 60°C
    .withColumn("humidity",    F.round(F.rand() * 60 + 30, 2))           # 30% to 90%
    .withColumn("location",    F.element_at(F.array(*[F.lit(l) for l in LOCATIONS]), (F.col("value") % len(LOCATIONS) + 1).cast("int")))
    .withColumn("event_time",  F.col("timestamp"))
    .select("sensor_id", "temperature", "humidity", "location", "event_time")
)

# Write to Polaris-backed Iceberg table
write_query = (
    sensor_stream
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/sensor_events")
    .toTable("polaris.streaming.sensor_events")
)

print("Streaming write started — ingesting sensor data for 30 seconds...")
time.sleep(30)
write_query.stop()
print(f"Streaming write stopped. Status: {write_query.status}")

## 4. Verify Written Data
Query the Iceberg table to see the rows that were streamed in.

In [ ]:
events = spark.table("polaris.streaming.sensor_events")
print(f"Total rows written: {events.count()}")
events.orderBy(F.desc("event_time")).show(15, truncate=False)

In [ ]:
# Per-sensor statistics
events.groupBy("sensor_id", "location") \
    .agg(
        F.count("*").alias("readings"),
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.round(F.min("temperature"), 2).alias("min_temp"),
        F.round(F.max("temperature"), 2).alias("max_temp"),
        F.round(F.avg("humidity"), 2).alias("avg_humidity")
    ) \
    .orderBy("sensor_id") \
    .show()

In [ ]:
# Iceberg snapshot history — each micro-batch creates a snapshot
spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records'] AS added_records,
           summary['total-records'] AS total_records,
           summary['added-data-files'] AS added_files
    FROM polaris.streaming.sensor_events.snapshots
    ORDER BY committed_at
""").show(truncate=False)

## 5. Streaming Read — Incremental Processing from Iceberg

Iceberg supports **incremental reads** — processing only newly appended data.
We read from `sensor_events`, filter for high-temperature alerts, and write
the results to `sensor_alerts`.

First, let's start another write stream to produce data, then read it incrementally.

In [ ]:
import time

# Start a new write stream to produce data while we read
raw_stream2 = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
sensor_stream2 = (
    raw_stream2
    .withColumn("sensor_id",   F.element_at(F.array(*[F.lit(s) for s in SENSORS]),   (F.col("value") % len(SENSORS) + 1).cast("int")))
    .withColumn("temperature", F.round(F.rand() * 50 + 10, 2))
    .withColumn("humidity",    F.round(F.rand() * 60 + 30, 2))
    .withColumn("location",    F.element_at(F.array(*[F.lit(l) for l in LOCATIONS]), (F.col("value") % len(LOCATIONS) + 1).cast("int")))
    .withColumn("event_time",  F.col("timestamp"))
    .select("sensor_id", "temperature", "humidity", "location", "event_time")
)
producer = (
    sensor_stream2.writeStream.format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/sensor_events_2")
    .toTable("polaris.streaming.sensor_events")
)

# Read incrementally from the Iceberg table
incremental_df = (
    spark.readStream
    .format("iceberg")
    .option("streaming-skip-delete-snapshots", "true")
    .option("streaming-skip-overwrite-snapshots", "true")
    .load("polaris.streaming.sensor_events")
)

# Filter for high-temperature alerts (> 35°C)
alerts_df = (
    incremental_df
    .filter(F.col("temperature") > 35)
    .withColumn("alert_type", F.lit("HIGH_TEMPERATURE"))
    .select("sensor_id", "temperature", "humidity", "location", "event_time", "alert_type")
)

# Write alerts to a separate Iceberg table
alert_query = (
    alerts_df
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/sensor_alerts")
    .toTable("polaris.streaming.sensor_alerts")
)

print("Producer + alert pipeline running for 20 seconds...")
time.sleep(20)
alert_query.stop()
producer.stop()
print("Both streams stopped.")

## 6. Verify Alerts
Check the `sensor_alerts` table for high-temperature readings.

In [ ]:
alerts = spark.table("polaris.streaming.sensor_alerts")
print(f"Total alerts: {alerts.count()}")
alerts.orderBy(F.desc("temperature")).show(15, truncate=False)

In [ ]:
# Alert summary by sensor and location
alerts.groupBy("sensor_id", "location") \
    .agg(
        F.count("*").alias("alert_count"),
        F.round(F.avg("temperature"), 2).alias("avg_alert_temp"),
        F.round(F.max("temperature"), 2).alias("max_temp")
    ) \
    .orderBy(F.desc("alert_count")) \
    .show()

## 7. Advanced: Partitioned Writes with Fanout

For partitioned Iceberg tables, Spark requires data sorted by partition key.
In streaming, use `fanout-enabled` to let Spark handle partitions automatically
without pre-sorting.

```python
# Fanout writing for partitioned tables
query = (
    input_stream
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("fanout-enabled", "true")
    .option("checkpointLocation", "/tmp/checkpoints/partitioned")
    .toTable("polaris.streaming.sensor_events")
)
```

### Handling Deletes and Overwrites

By default, Iceberg streaming does not process delete or overwrite snapshots.
If your source table receives compaction or rewrites, add these options to skip them:

```python
streaming_df = (
    spark.readStream
    .format("iceberg")
    .option("streaming-skip-overwrite-snapshots", "true")
    .option("streaming-skip-delete-snapshots", "true")
    .load("polaris.streaming.sensor_events")
)
```

### Starting from a Specific Timestamp

Use `stream-from-timestamp` (epoch milliseconds) to start reading from a point in time:

```python
streaming_df = (
    spark.readStream
    .format("iceberg")
    .option("stream-from-timestamp", "1640995200000")  # Jan 1, 2022
    .load("polaris.streaming.sensor_events")
)
```

## 8. Maintenance — Compact Small Files & Expire Snapshots

Streaming writes create many small files (one per micro-batch per partition).
Regular maintenance is essential for query performance.

In [ ]:
# Check current file count and sizes
files_df = spark.sql("""
    SELECT file_format, count(*) as file_count,
           sum(record_count) as total_records,
           round(sum(file_size_in_bytes) / 1024, 2) as total_size_kb
    FROM polaris.streaming.sensor_events.files
    GROUP BY file_format
""")
files_df.show()

In [ ]:
# Compact small files into larger ones
spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'streaming.sensor_events',
        options => map('min-input-files', '2')
    )
""").show(truncate=False)

print("Files after compaction:")
spark.sql("""
    SELECT count(*) as file_count, sum(record_count) as total_records,
           round(sum(file_size_in_bytes) / 1024, 2) as total_size_kb
    FROM polaris.streaming.sensor_events.files
""").show()

In [ ]:
# Expire old snapshots to reclaim storage
from datetime import datetime, timedelta

cutoff = (datetime.now() - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
print(f"Expiring snapshots older than: {cutoff}")

spark.sql(f"""
    CALL polaris.system.expire_snapshots(
        table => 'streaming.sensor_events',
        older_than => TIMESTAMP '{cutoff}',
        retain_last => 2
    )
""").show(truncate=False)

In [ ]:
# Optimize manifest files
spark.sql("""
    CALL polaris.system.rewrite_manifests(
        table => 'streaming.sensor_events'
    )
""").show(truncate=False)

## 9. Cleanup
Drop the streaming tables and namespace.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.streaming.sensor_alerts PURGE")
spark.sql("DROP TABLE IF EXISTS polaris.streaming.sensor_events PURGE")
spark.sql("DROP NAMESPACE IF EXISTS polaris.streaming")
print("Cleanup complete.")
spark.sql("SHOW NAMESPACES IN polaris").show()

## Summary

| Pattern | What it does |
|---------|-------------|
| `readStream.format("rate")` → `writeStream.format("iceberg").toTable(...)` | Ingest streaming data into Polaris |
| `readStream.format("iceberg").load(...)` | Incrementally read new Iceberg snapshots |
| `streaming-skip-overwrite-snapshots` | Skip compaction snapshots in reads |
| `fanout-enabled` | Auto-handle partitions in streaming writes |
| `stream-from-timestamp` | Start stream from a specific point in time |
| `rewrite_data_files` | Compact small files from micro-batches |
| `expire_snapshots` | Reclaim storage from old snapshots |
| `rewrite_manifests` | Optimize Iceberg manifest files |